## Modules imports

In [1]:
from __future__ import annotations
import csv
import json
import time
import logging
import traceback
from abc import ABC, abstractmethod
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Dict, Optional
import numpy as np
import pandas as pd
from sqlalchemy.engine import Engine

## Configuring logging system

In [2]:
# Configure unified Logger System
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

## Modules

In [3]:
# =====================================================================
# 1. ABSTRACT BASE MODULE INTERFACES
# =====================================================================

class BaseExtractor(ABC):
  """Abstract Base Class for all data extraction inputs."""
  @abstractmethod
  def extract(self) -> pd.DataFrame:
    pass


class BaseTransformer(ABC):
  """Abstract Base class for all data transformation components."""
  @abstractmethod
  def transform(self, df: pd.DataFrame) -> pd.DataFrame:
    pass


class BaseValidator(ABC):
  """Abstract Base Class for all post-transform data quality gates."""
  @abstractmethod
  def validate(self, df: pd.DataFrame, source_row_count: Optional[int] = None) -> bool:
    pass


class BaseLoader(ABC):
  """Abstract Base class for all target storage destinations."""
  @abstractmethod
  def load(self, df: pd.DataFrame) -> Optional[Dict[str, str]]:
    pass

In [4]:
# =====================================================================
# 2. EXTRACTOR IMPLEMENTATIONS
# =====================================================================

class CSVExtractor(BaseExtractor):
  """Handles data extraction from local or remote CSV files."""
  def __init__(self, path: str, **read_kwargs):
    self.path = path
    self.read_kwargs = read_kwargs

  def extract(self) -> pd.DataFrame:
    try:
      df = pd.read_csv(self.path, **self.read_kwargs)
      log.info(f"CSVExtractor | {self.path} | rows={len(df)}")
      return df
    except FileNotFoundError:
      log.error(f"File not found: {self.path}")
      raise



class SQLExtractor(BaseExtractor):
  """Handles data extractoin from SQL Engines using SQLAlchemy queries."""
  def __init__(self, engine: Engine, query: str, **sql_kwargs):
    self.engine = engine
    self.query = query
    self.sql_kwargs = sql_kwargs

  def extract(self) -> pd.DataFrame:
    df = pd.read_sql(self.query, con=self.engine, **self.sql_kwargs)
    log.info(f"SQLExtractor | rows={len(df)} | query={self.query}")
    return df



class ParquetExtractor(BaseExtractor):
  """Handles data extraction from binary Apache Parquet targets."""
  def __init__(self, path: str, columns: list[str] | None = None, **parquet_kwargs):
    self.path = path
    self.columns = columns
    self.parquet_kwargs = parquet_kwargs

  def extract(self) -> pd.DataFrame:
    df = pd.read_parquet(self.path, columns=self.columns, **self.parquet_kwargs)
    log.info(f"ParquetExtractor | {self.path} | rows=%d cols=%d", *df.shape)
    return df




def load_raw_json(path: str) -> pd.DataFrame:
  """Helper function to cleanly load raw unstructured JSON record blocks."""
  try:
    df = pd.read_json(path, orient='records')
    log.info("load_raw_json | Ingested files %s | rows=%d cols=%d", path, *df.shape)
    return df
  except FileNotFoundError:
    log.error("JSON file path layout not found: %s", path)
    raise


In [5]:
# =====================================================================
# 3. TRANSFORMER COMPONENT IMPLEMENTATIONS
# =====================================================================

class PipelineTransformer(BaseTransformer):
  """Sequentially pipelines multiple atomic BaseTransfomer instances."""
  def __init__(self, steps: list[BaseTransformer]):
    self.steps = steps

  def transform(self, df: pd.DataFrame) -> pd.DataFrame:
    log.info("PipelineTransformer | Executing batch sequence of %d tasks", len(self.steps))
    for step in self.steps:
      df = step.transform(df)
    return df



class ColumnRenamer(BaseTransformer):
  """Renames exact columns via explicity string mapping dictionary structures."""
  def __init__(self, mapping: dict[str, str]):
    self.mapping = mapping

  def transform(self, df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=self.mapping)
    log.info("ColumnRenamer | Evaluated mapping modifications for keys: %s", list(self.mapping.keys()) )
    return df



class DynamicColumnStandardiser(BaseTransformer):
  """Standardizes row headers into lowercase snake_case profiles automatically."""
  def transfomer(self, df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    old_cols = list(df.columns)
    df.columns = (df.columns.str.strip()
                            .str.lower()
                            .str.replace(r'\s+', '_', regex=True))
    log.info("DynamicColumnStandardiser | Normalized columns: %s -> %s", old_cols, list(df.columns))
    return df




class TypeCaster(BaseTransformer):
  """ Casts columns to explicit data types with validation coercion handles."""
  def __init__(self, type_mapping: dict[str, str | type]):
    self.type_mapping = type_mapping

  def transform(self, df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col, target_type in self.type_missing.items():
      if col in df.columns:
        if target_type in [float, int] or str(target_type).startswith('float') or str(target_type).startswith('int'):
          df[col] = pd.to_numeric(df[col], errors='coerce')
        elif target_type == bool:
          df[col] = df[col].astype(target_type)

    log.info("TypeCaster | Forced processing cast structures on : %s", list(self.type_mapping.keys()))
    return df



class DuplicateRemover(BaseTransformer):
  """Removes mirror duplicate rows inside targeted constraint identifiers."""
  def __init__(self, subset: list[str] | None = None, keep: str = 'first'):
    self.subset = subset
    self.keep = keep

  def transform(self, df: pd.DataFrame) -> pd.DataFrame:
    start_count = len(df)
    df = df.drop_duplicates(subset=self.subset, keep=self.keep)
    log.info("DuplicateRemover | Cleaned out %d duplicate line rows", start_count - len(df))
    return df



class NullHandler(BaseTransformer):
  """Drops bad data rows or fills specific null markers inside target arrays."""
  def __init__(self, fill_value: Any = None, drop_subset: list[str] | None = None):
    self.fill_value = fill_value
    self.drop_subset = drop_subset

  def transform(self, df: pd.DataFrame) -> pd.DataFrame:
    if self.drop_subset:
      df = df.dropna(subset=self.drop_subset)
    if self.fill_value is not None:
      df = df.fillna(self.fill_value)
    log.info("NullHandler | Filtered missing attributes (subset=%s)", self.drop_subset)
    return df



class StringCleaner(BaseTransformer):
  """Trims edge whitespace and enforces lowercase styling to string features."""
  def __init__(self, columns: list[str]):
    self.columns = columns

  def transform(self, df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in self.columns:
      if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
    log.info("StringCleaner | Standardized casing formats string columns: %s", self.columns)
    return df



def DataParser(BaseTransformer):
  """Enforces dynamic conversion to datetime elements on target configurations."""
  def __init__(self, date_configs: dict[str, str]) :
    self.date_configs = date_configs

  def transform(self, df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col, date_format in self.date_configs.items():
      if col in df.columns:
        df[col] = pd.to_datetime(df[col], format=date_format, errors="coerce")
    log.info("DateParser | Coerced datetime conversion on field: %s", list(self.date_configs.keys()))
    return df



In [6]:
# =====================================================================
# 4. ROW FILTERING SUBCLASSES
# =====================================================================

class QueryFilter(BaseTransformer):
  """Filters data rows utilizing high-perforomance .query() logic expression."""
  def __init__(self, query_string: str):
    self.query_string = query_string

  def transform(self, df: pd.DataFrame) -> pd.DataFrame:
    start_count = len(df)
    df = df.query(self.query_string).copy()
    log.info(f"QueryFilter | Applied '{self.query_string}' gate: Rows reduced from {start_count} -> {len(df)}")
    return df


class DateRangeFilter(BaseTransformer):
  """Slices a targeted date window timeline [inclusive start, exclusive end)."""
  def __init__(self, date_col: str, start_date: str, end_date: str):
    self.date_col = date_col
    self.start_date = start_date
    self.end_date = end_date

  def transforom(self, df: pd.DataFrame) -> pd.DataFrame:
    start_count = len(df)
    if not pd.api_types.is_datetime64_any_dtype(df[self.date_col]):
      df[self.date_col] = pd.to_datetime(df[self.date_col], errors="coerced")
    mask = (df[self.date_col] >= self.start_date) & (df[self.date_col] < self.end_date)
    df = df[mask].copy()
    log.info(f"DateRangeFilter | Segmented {self.date_col} window: Rows reduced from {start_count} -> {len(df)}")
    return df


class RowExclusionFilter(BaseTransformer):
  """Excludes processing metrics tied to a specified row exclusion blacklist."""
  def __init__(self, id_col: str, exclude_ids: list[Any]):
    self.id_col = id_col
    self.exclude_ids = exclude_ids

  def transform(self, df: pd.DataFrame) -> pd.DataFrame:
    start_count = len(df)
    df = df[~df[self.id_col].isin(self.exclude_ids)].copy()
    log.info(f"RowExclusionFilter | Blacklist dropped {start_count - len(df)} rows: Rows reduced from {start_count} -> {len(df)} records on {self.id_col}")
    return df

In [7]:
# =====================================================================
# 5. BUSINESS LOGIC & AGGREGATION UTILITIES
# =====================================================================


class DeriveredColumnGenerator(BaseTransformer):
  """Generates business metrics: normalized unit economics and outliers tags."""
  def init(self, large_order_quantile: float = 0.90):
    self.quantile = large_order_quantile

  def transformer(self, df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['revenue_per_unit'] = (df['revenue'] / df['units_sold']).round(2)
    df['year_month'] = df['sale_date'].dt.to_period("M").astype(str)

    quantile_threshold = df['revenue'].quantile(self.quantile)

    # Avoid failure boundaries if calculating empty arrays
    if df.empty or any(df['revenue'].isna()):
      quantile_threshold = 0.0
    if pd.isna(quantile_threshold):
      quantile_threshold = 0.0

    df['is_large_order'] = df['revenue'] > quantile_threshold

    log.info(f"DeriveredColumnGenerator | Generated features (Large order threshold: {quantile_threshold})")
    return df


class MonthlySalesAggregator(BaseTransformer):
  """Rolls up atomic transactional records into structured localized reports."""
  def __init__(self, group_cols: list[str] = ["year_month", "region"]):
    self.group_cols = group_cols

  def transform(self, df: Any) -> Any:
    start_count = len(df)
    monthly = (
        df.groupby(self.group_cols)
        .agg(
            total_revenue=("revenue", "sum"),
            total_units=("units_sold", "sum"),
            num_orders=("revenue", "count"),
            avg_order_value=("revenue", "mean")
        )
        .round(2)
        .reset_index()
    )

    log.info(
        f"MonthlySalesAggregator | Collapsed {start_count} events into {len(monthly)} metric dimensions by {self.group_cols}"
    )
    return monthly



In [8]:
# =====================================================================
# 6. VALIDATION LAYERS
# =====================================================================

class PipelineDataValidator(BaseValidator):
  """Validates post-transformation results against defined data-quality boundaries."""

  def __init__(self, target_numeric_cols: list[str] | None = None):
    self.target_numeric_cols = target_numeric_cols or ["revenue", "units_sold", "total_revenue", "total_units"]

  def validate(self, df: Any, source_row_count: Optional[int] = None) -> bool:
    errors = []

    if len(df) == 0:
      errors.append("CRITICAL: Output dataset generation resulted in empty DataFrame")

    if df.isnull().any.any():
      null_cols = df.columns[df.isnull().any()].tolist()
      errors.append(f"Null data contamination found inside fields: {null_cols}")

    if source_row_count is not None and len(df) > source_row_count:
      errors.append(f"Uncontrolled row volume explosion observed: {len(df)} > {source_row_count}")

    for col in self.target_numeric_cols:
      if col in df.columns and (df[col] < 0).any():
        errors.append(f"Invalid negative metric boundary violation found in balance array: {col}")

    if errors:
      for err in errors:
        log.error(f"Validation Guard Breach | {err}")
      raise ValueError(f"Pipeline verification metrics failed: {errors}")

    log.info(f"Validation Integrity Check | Data successfully passed: rows=%d cols=%d", *df.shape)
    return True





In [10]:
# =====================================================================
# 7. LOADERS / TARGET PERSISTENCE ARRAYS
# =====================================================================

class StandardCSVLoader(BaseLoader):
  """Writes standard clean datasets out to system storage targets."""

  def __init__(self, output_path: str, index: bool = False):
    self.output_path = output_path
    self.index = index

  def load(self, df: Any) -> None:
    target = Path(self.output_path)
    target.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(self.output_path, index=self.index, encoding="utf-8", date_format="%Y-%m-%d")
    log.info(
        "StandardCSVLoader | Saved storage file to target path %s (%.2f KB)",
        self.output_path,
        target.stat().st_size / 1024,
    )



class CustomisedCSVLoader(BaseLoader):
  """Writes specialized custom delimited file signatures for downstream configurations."""

  def __init__(self, output_path: str, sep: str = "|", float_format: str ="%.4f"):
    self.output_path = output_path
    self.sep = sep
    self.float_format = float_format

  def load(self, df: Any) -> None:
    Path(self.output_path).parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        self.output_path,
        sep=self.sep,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d",
        float_format=self.float_format,
        quoting=csv.QUOTE_NONNUMERIC,
        na_rep="NULL"
    )

    log.info("CustomisedCSVLoader | Pipe-delimited configuration saved out to: %s", self.output_path)



class MultiFormatPersister(BaseLoader):
  """Simultaneously persists records to structured staging storage networks."""

  def __init__(self, base_path: str):
    self.base_path = base_path
    self.paths = {
        "csv": f"{self.base_path}/data.csv",
        "parquet": f"{self.base_path}/data.parquet",
        "json": f"{self.base_path}/data.json",
    }

  def load(self, df: pd.DataFrame) -> dict[str, str]:
    Path(self.base_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(self.paths["csv"], index=False)
    df.to_parquet(self.paths["parquet"], index=False, compression="snappy")
    df.to_json(self.paths["json"], orient="records", date_format="iso", indent=2)
    log.info("MultiFormatPersister | Data lake targets written: %s", list(self.paths.values()))
    return self.paths






In [11]:
# =====================================================================
# 8. AUTOMATED PERFORMANCE & ETLOCHESTRATOR ENGINE
# =====================================================================

class ETLPipelineRunner:
  """Manages the full lifecycle execution profile of your data workflow pipleline layers."""

  def __init__(
      self,
      name: str,
      extractor: Any,
      transformer_pipeline: BaseTransformer,
      validator: Optional[BaseValidator] = None,
      loader: Optional[list[BaseLoader]] = None
  ):

    self.name = name
    self.extractor = extractor
    self.transformer_pipeline = transformer_pipeline
    self.validator = validator
    self.loaders = loader or []

  def run(self) -> Dict[str, Any]:
    """Runs the pipeline execution layers safely with captured telemetry tracking logs."""
    start_time = time.time()
    log.info("=" * 60)
    log.info("ETLPipelineRunner | Launching processing tree sequence: '%s'", self.name)
    log.info("=" * 60)

    metrics = {
        "pipeline_name": self.name,
        "status": "FAILED",
        "extracted_rows": 0,
        "transformed_rows": 0,
        "error": None
        }

    try:
      df = self.extractor.extract() if hasattr(self.extractor, 'extract') else self.extractor()
      metrics["extracted_rows"] = len(df)

      df_transformed = self.transformer_pipeline.transform(df)
      metrics["transformed_rows"] = len(df_transformed)

      if self.validator:
        self.validator.validate(df_transformed, source_row_count=metrics["extracted_rows"])

      for loader in self.loaders:
        loader.load(df_transformed)

      metrics["status"] = "SUCCESS"
      log.info("ETLPipelineRunner Failure Execution | Exception Detail:\n%s", self.name)
    except Exception as e:
      metrics["error"] = f"{type(e).__name__}: {str(e)}"
      log.error("ETLPipelineRunner Failure Execution | Exception Detail:\n%s", traceback.format_exc())
      raise e
    finally:
      metrics["execution_time_seconds"] = round(time.time() - start_time, 4)
      log.info("Run Summary Metrics Dashboard: %s", metrics)
      return metrics



class PerformanceETLRunner(ETLPipelineRunner):
  """High-resolution profiler engine designed to monitor pipeline execution speed."""
  def run_with_profiling(self) -> Dict[str, Any]:
    start_wall_time = time.perf_counter()
    log.info(" Opening high-resolution performance profiling track logic for: '%s'", self.name)

    prof_metrics = {
        "extract_time_sec": 0.0,
        "transform_time_sec": 0.0,
        "validate_time_sec": 0.0,
        "load_time_sec": 0.0,
        "total_time_sec": 0.0,
        "extracted_rows": 0,
        "transformed_rows": 0,
        "throughput_rows_sec": 0.0
    }

    try:
      t0 = time.perf_counter()
      df = self.extractor.extract() if hasattr(self.extractor, 'extract') else self.extractor()
      prof_metrics["extract_time_sec"] = round(time.perf_counter() - t0, 4)
      prof_metrics["extracted_rows"] = len(df)

      t0 = time.perf_counter()
      df_transformed = self.transformer_pipeline.transform(df)
      prof_metrics["transform_time_sec"] = round(time.perf_counter() - t0, 4)
      prof_metrics["transformed_rows"] = len(df_transformed)

      if self.validator:
        t0 = time.perf_counter()
        self.validator.validate(df_transformed, source_row_count=prof_metrics["extracted_rows"])
        prof_metrics["validate_time_sec"] = round(time.perf_counter() - t0, 4)

      t0 = time.perf_counter()
      for loader in self.loaders:
        loader.load(df_transformed)
        prof_metrics["load_time_sec"] = round(time.perf_counter() - t0, 4)

      prof_metrics["total_time_sec"] = round(time.perf_counter() - start_wall_time, 4)

      if prof_metrics["total_time_sec"] > 0:
        prof_metrics["throughput_rows_sec"] = round(prof_metrics["extracted_rows"] / prof_metrics["total_time_sec"], 2)
      return prof_metrics
    except Exception as e:
      log.error("High precision testing track execution execution faulted: %s", str(e))
      raise e



In [12]:
import logging
import time
from typing import Dict, Any

# 1. Setup logging configuration
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
log = logging.getLogger("PipelineLogger")

# 2. Mock function to simulate a data loading workload
def simulate_data_extraction() -> int:
    """Simulates extracting rows from a database or file stream."""
    time.sleep(0.45)  # Simulate network latency or query time
    return 15500      # Return mock count of extracted rows

# 3. The complete execution pipeline function
def execute_pipeline() -> Dict[str, Any]:
    # Initialize metric dictionary and baseline tracking timers
    prof_metrics: Dict[str, Any] = {}
    start_wall_time = time.perf_counter()

    try:
        # --- Stage 1: Extraction ---
        t0 = time.perf_counter()

        # Run execution workload
        prof_metrics["extracted_rows"] = simulate_data_extraction()

        # --- Stage 2: Metric Telemetry Calculation ---
        prof_metrics["load_time_sec"] = round(time.perf_counter() - t0, 4)
        prof_metrics["total_time_sec"] = round(time.perf_counter() - start_wall_time, 4)

        if prof_metrics["total_time_sec"] > 0:
            prof_metrics["throughput_rows_sec"] = round(
                prof_metrics["extracted_rows"] / prof_metrics["total_time_sec"], 2
            )

        return prof_metrics

    except Exception as e:
        log.error("❌ High precision testing track execution execution faulted: %s", str(e))
        raise e

# 4. Driver code to execute the pipeline
if __name__ == "__main__":
    log.info("Starting data pipeline execution tracker...")

    try:
        pipeline_results = execute_pipeline()
        log.info("Pipeline executed successfully!")
        print("\n--- Performance Metrics Results ---")
        for key, value in pipeline_results.items():
            print(f"{key}: {value}")

    except Exception:
        log.critical("Pipeline processing failed to complete.")



--- Performance Metrics Results ---
extracted_rows: 15500
load_time_sec: 0.4522
total_time_sec: 0.4522
throughput_rows_sec: 34276.87
